Imports and dates

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Add repo root to PYTHONPATH
HERE = Path().resolve()
PROJECT_ROOT = HERE.parents[0]   # notebooks → repo root
sys.path.insert(0, str(PROJECT_ROOT))


from prm_opt.run_s25 import run_s25_s1, run_s25_s2, run_s25_s2_v2
from prm_opt.run_s26 import run_s26_s1, run_s26_s2
from prm_opt.outputs import build_run_report, print_run_report, build_run_report_s1, print_run_report_s1
from prm_opt.build_jobs import build_jobs
from prm_opt.ingest_s25 import ingest_s25
from prm_opt.config import PlanningToggles

# -----------------------------
# DATE RANGES
# -----------------------------
START_S25 = "2025-03-30"
END_S25   = "2025-10-26"

START_S25_DEBUG = "2025-03-30"
END_S25_DEBUG   = "2025-04-02"

START_S26 = "2025-03-29"
END_S26   = "2025-10-24"

Common toggles


In [2]:

toggles = PlanningToggles(
    sla_buffer_mins=0,
    spill_bucket_cap=12,
    standby_dep_vert_mins=10,
    standby_arr_horiz_mins=10,
)


Run S25 Scenario 1 (baseline)

In [3]:

out_s1 = run_s25_s1(START_S25, END_S25, toggles=toggles)

report_s1 = build_run_report_s1(out_s1, day_from="s", hour_method="max")
print_run_report_s1(report_s1)

# # 15-min peak day detail:
# report_s1["peak_day_report"]["bucket_level"].to_csv("S1_peak_day_15min.csv")
# report_s1["peak_day_report"]["hourly"].to_csv("S1_peak_day_hourly.csv")





Unmatched passenger rows after merge (missing Chocks DT): 4437
Unique unmatched flight keys: 2699

Unmatched key reasons:
reason
no_flight_candidate               1935
scheduled_dt_mismatch              702
exact_match_should_have_joined      62
Name: count, dtype: int64

Sample unmatched keys with reasons (top 50):
     Airline Code Flight Number A/D Scheduled Flight DT_prm  \
1749           AA           279   D     2025-08-09 11:35:00   
1792           AA           279   D     2025-08-13 11:35:00   
2676           AF          1887   D     2025-10-21 06:00:00   
555            B6            72   D     2025-05-14 11:30:00   
1396           B6          1080   D     2025-07-15 12:10:00   
2212           B6          1080   D     2025-09-14 12:10:00   
779            BA          1459   D     2025-05-29 19:05:00   
2692           BA          1457   D     2025-10-23 08:35:00   
1674           D8          3503   D     2025-08-04 15:10:00   
2479           D8          4464   D     2025-10-03 

C:\Users\jamie_douglas\OneDrive - Edinburgh Airport Limited\Documents\GitHub\EDI_airport_analytics\prm_opt\outputs.py:562: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly = bucket_level.resample("H").max()


Run S25 Scenario 2 (optimised)

In [4]:

#out = run_s25_s2_lp(START_S25, END_S25, toggles=toggles, run_lp_ladder=True, solve_milp=False)
#out["lp_baseline"], out["lp_ladder"]


In [ ]:

out = run_s25_s2_v2(
    start=START_S25,
    end=END_S25,
    solver_name="highs",
    toggles=toggles,
    solve_model=True,
    time_limit_sec=600,
    threads=8,
    mip_rel_gap=0.20,  # optional: stop earlier with a usable solution
)

report = build_run_report(out)     # builds sanity + peak day hourly
print_run_report(report)           # prints clean report


# report["peak_day_report"]["hourly"].to_csv("peak_day_hourly_fleet.csv")


PRM OPT — S25 Scenario 2 (v2)
Window : 2025-03-30 → 2025-10-26

[1/6] ingest_s25…

Unmatched passenger rows after merge (missing Chocks DT): 4437
Unique unmatched flight keys: 2699

Unmatched key reasons:
reason
no_flight_candidate               1935
scheduled_dt_mismatch              702
exact_match_should_have_joined      62
Name: count, dtype: int64

Sample unmatched keys with reasons (top 50):
     Airline Code Flight Number A/D Scheduled Flight DT_prm  \
1749           AA           279   D     2025-08-09 11:35:00   
1792           AA           279   D     2025-08-13 11:35:00   
2676           AF          1887   D     2025-10-21 06:00:00   
555            B6            72   D     2025-05-14 11:30:00   
1396           B6          1080   D     2025-07-15 12:10:00   
2212           B6          1080   D     2025-09-14 12:10:00   
779            BA          1459   D     2025-05-29 19:05:00   
2692           BA          1457   D     2025-10-23 08:35:00   
1674           D8          3503

In [ ]:
# out_s25_s2 = run_s25_s2(
#     start=START_S25,
#     end=END_S25,
#     solver_name="highs",
#     toggles=toggles,
#     run_ladder=True, 
#     solve_model=False
# )

# out_s25_s2["summary"]

Run S26 Scenario 1

In [ ]:

out_s26_s1 = run_s26_s1(
    start=START_S26,
    end=END_S26,
    penetration_rates=penetration_rates,
    ssr_mix=ssr_mix,
    stand_actuals=stand_actuals,
    stand_dist=stand_dist,
    service_time_params=service_time_params,
    chocks_offset_params=chocks_offset_params,
    toggles=toggles,
)

out_s26_s1["summary"]
out_s26_s1["ambulift_curve"].head()
out_s26_s1["driver_curve"].head()


Run S26 scenario 2

In [ ]:

out_s26_s2 = run_s26_s2(
    start=START_S26,
    end=END_S26,
    penetration_rates=penetration_rates,
    ssr_mix=ssr_mix,
    stand_actuals=stand_actuals,
    stand_dist=stand_dist,
    service_time_params=service_time_params,
    chocks_offset_params=chocks_offset_params,
    solver_name="highs",
    toggles=toggles,
)

out_s26_s2["summary"]
